In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import json
from kimina_client import KiminaClient

/workspace/llm-autoformalization/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "AI-MO/Kimina-Autoformalizer-7B"
model = LLM(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
sampling_params = SamplingParams(temperature=0.6, top_p=0.95, max_tokens=2048)

INFO 04-27 09:58:48 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'AI-MO/Kimina-Autoformalizer-7B'}


INFO 04-27 09:58:49 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-27 09:58:49 [model.py:1678] Using max model len 32768
INFO 04-27 09:58:49 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-27 09:58:49 [vllm.py:790] Asynchronous scheduling is enabled.
INFO 04-27 09:58:52 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='AI-MO/Kimina-Autoformalizer-7B', speculative_config=None, tokenizer='AI-MO/Kimina-Autoformalizer-7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=Str

(EngineCore pid=33228) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=33228) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:01,  1.60it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.17it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:02<00:00,  1.12it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:03<00:00,  1.38it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:03<00:00,  1.31it/s]
(EngineCore pid=33228) 


INFO 04-27 09:58:58 [default_loader.py:384] Loading weights took 3.14 seconds
INFO 04-27 09:58:59 [gpu_model_runner.py:4820] Model loading took 14.25 GiB memory and 4.322214 seconds
INFO 04-27 09:59:00 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/b69cc73675/rank_0_0/backbone for vLLM's torch.compile
INFO 04-27 09:59:00 [backends.py:1111] Dynamo bytecode transform time: 1.56 s
INFO 04-27 09:59:02 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.129 s
INFO 04-27 09:59:02 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/568f230f816c22da8c64218547c0c12587289ccc424da21d012de6f47872e3c4/rank_0_0/model
INFO 04-27 09:59:02 [monitor.py:48] torch.compile took 2.88 s in total
INFO 04-27 09:59:02 [monitor.py:76] Initial profiling/warmup run took 0.13 s
INFO 04-27 09:59:08 [kv_cache_utils.py:829] Overriding num_gpu_blocks=0 with num_gpu_blocks

(EngineCore pid=33228) 2026-04-27 09:59:10,166 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=33228) 2026-04-27 09:59:10,176 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 28.38it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 38.35it/s]


INFO 04-27 09:59:13 [gpu_model_runner.py:6046] Graph capturing finished in 4 secs, took 0.35 GiB
INFO 04-27 09:59:13 [gpu_worker.py:597] CUDA graph pool memory: 0.35 GiB (actual), 0.38 GiB (estimated), difference: 0.03 GiB (8.9%).
INFO 04-27 09:59:13 [core.py:283] init engine (profile, create kv cache, warmup model) took 14.71 seconds


In [3]:
import json

data_path = "../datasets/formallite_combibench_proverbench.jsonl"

data = []
with open(data_path, "rb") as f:
    for line in f.readlines():
        data.append(
            json.loads(line)
        )

In [4]:
def prepare_prompt(problem: str):
    prompt = "Please autoformalize the following problem in Lean 4 with a header. Use the following theorem names: my_favorite_theorem.\n\n"
    prompt += problem

    messages = [
        {"role": "system", "content": "You are an expert in mathematics and Lean 4."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return text

In [5]:
text = prepare_prompt(data[10]["problem"])
print(text)

<|im_start|>system
You are an expert in mathematics and Lean 4.<|im_end|>
<|im_start|>user
Please autoformalize the following problem in Lean 4 with a header. Use the following theorem names: my_favorite_theorem.

Proof: $\sum_{k = 0}^{\left[\frac{n - 1}{2}\right]}(-1)^k\mathrm{C}_{n + 1}^k\mathrm{C}_{2n - 2k - 1}^n=\frac{1}{2}n(n + 1)\ (n\geqslant1)$<|im_end|>
<|im_start|>assistant



In [6]:
output = model.generate(text, sampling_params=sampling_params)
output_text = output[0].outputs[0].text
print(output_text)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it, est. speed input: 77.05 toks/s, output: 60.46 toks/s]


import Mathlib

open Finset Real Nat

theorem my_favorite_theorem {n : ℕ} (hn : 1 ≤ n) :
    ∑ k in range ((n - 1) / 2 + 1), (-1 : ℝ)^k * Nat.choose (n + 1) k * Nat.choose (2 * n - 2 * k - 1) n =
    (1 / 2) * n * (n + 1) := by sorry


In [7]:
client = KiminaClient()

In [8]:
response = client.check(output_text)
response.analyze(elapsed=1)

Batches: 100%|██████████| 1/1 [elapsed: 00:00, remaining: 00:00]



╒═════╤════════════╤════════════╤═════════════════╤══════════════╤══════════════╤════════════════╤══════════════════╤════════════════╤═══════════╕
│   # │   Valid ✅ │   Sorry ⚠️ │  Lean Error ❌  │   Timeout ⏰ │   REPL Error │   Server Error │  Total CPU Time  │  Avg CPU Time  │  Elapsed  │
╞═════╪════════════╪════════════╪═════════════════╪══════════════╪══════════════╪════════════════╪══════════════════╪════════════════╪═══════════╡
│   1 │          0 │          0 │    1 (100 %)    │            0 │            0 │              0 │      0.05 s      │ 0.05 s/snippet │  1.00 s   │
╘═════╧════════════╧════════════╧═════════════════╧══════════════╧══════════════╧════════════════╧══════════════════╧════════════════╧═══════════╛
